In [7]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [8]:
df = pd.read_csv("features (3).csv")
all_cols = [c for c in df.columns if c.startswith("feat_")]
print(df.shape)
print(df['class'].value_counts())

(294, 197)
class
Stone         161
Vegetation     77
Bare Soil      56
Name: count, dtype: int64


In [9]:
class_names = ["Bare Soil", "Stone", "Vegetation"]
class_to_int = {"Bare Soil": 0, "Stone": 1, "Vegetation": 2}

y = np.array([class_to_int[c] for c in df['class']])
X = df[all_cols].values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(len(class_names))
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 64)             │        12,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,739 (49.76 KB)

 Trainable params: 12,739 (49.76 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=2)

Epoch 1/10
8/8 - 2s - 281ms/step - accuracy: 0.4936 - loss: 1.1175
Epoch 2/10
8/8 - 0s - 21ms/step - accuracy: 0.7149 - loss: 0.7540
Epoch 3/10
8/8 - 0s - 16ms/step - accuracy: 0.7872 - loss: 0.5895
Epoch 4/10
8/8 - 0s - 58ms/step - accuracy: 0.8085 - loss: 0.5032
Epoch 5/10
8/8 - 0s - 57ms/step - accuracy: 0.8383 - loss: 0.4384
Epoch 6/10
8/8 - 0s - 17ms/step - accuracy: 0.8681 - loss: 0.3920
Epoch 7/10
8/8 - 0s - 28ms/step - accuracy: 0.9064 - loss: 0.3550
Epoch 8/10
8/8 - 0s - 34ms/step - accuracy: 0.9149 - loss: 0.3262
Epoch 9/10
8/8 - 0s - 50ms/step - accuracy: 0.9191 - loss: 0.3000
Epoch 10/10
8/8 - 1s - 75ms/step - accuracy: 0.9277 - loss: 0.2781


In [12]:
all_preds = model.predict(X_test, verbose=0).argmax(1)
all_true = y_test

acc = accuracy_score(all_true, all_preds)
prec = precision_score(all_true, all_preds, average='macro', zero_division=0)
rec = recall_score(all_true, all_preds, average='macro', zero_division=0)
f1 = f1_score(all_true, all_preds, average='macro', zero_division=0)

print(f'Accuracy : {acc:.3f}')
print(f'Precision: {prec:.3f}')
print(f'Recall   : {rec:.3f}')
print(f'F1 Score : {f1:.3f}')

print('Confusion matrix:')
print(class_names)
print(confusion_matrix(all_true, all_preds))

Accuracy : 0.847
Precision: 0.829
Recall   : 0.798
F1 Score : 0.811
Confusion matrix:
['Bare Soil', 'Stone', 'Vegetation']
[[ 7  3  1]
 [ 2 29  0]
 [ 1  2 14]]
